# value_cls 真牌微調 — 從現有權重出發，合成資料＋實拍錄影牌角混合訓練

現有 value_cls 只用合成資料訓練；實拍錄影（牌離鏡頭遠、牌角約 20px）常把 5♣ 認成 5♠、4♥ 認成 A♥，
也會把海報、牌中央的大花色讀成牌值。這裡把錄影裁出、人工複查過的牌角（含海報／書櫃／大花色等 none）
加進合成資料一起微調。為避免過擬合：真牌只佔一部分（過採樣倍數可調）、學習率調低、合成驗證集同時監看。

**Kaggle 設定**：GPU、Internet On、Add Input：
`gpiosenka/cards-image-datasetclassification`、`jmexpert/describable-textures-dataset-dtd`、
`yun193/cards-real-crops`（real/<類別>/*.jpg 與 base/ 或 base26/best.pt，依 BASE_DIR）

In [ ]:
N_TRAIN, N_VAL = 4000, 900     # 合成場景數；N_VAL 與 SEED 同原訓練 → 合成驗證集相同，可直接比 top-1
IMGSZ = 128
EPOCHS = 12
BATCH = 256
LR0 = 0.0005
REAL_POS_TARGET = 250          # 真牌牌值類別過採樣到約此張數（每類倍數 1~20，避免張數多的 6♦ 壓過其他類）
REAL_NONE_REPEAT = 2           # 真牌 none 過採樣倍數
NONE_RATIO = 1.5
SEED = 0
NB_TAG = "value26-ft-1"
BASE_DIR = "base26"             # 資料集裡的基底權重資料夾：base = v8（value-1）、base26 = YOLO26（value26-1）
OUT_NAME = "value26_ft"
print("NB_TAG=" + NB_TAG)

In [ ]:
import json, os, random, shutil
from collections import Counter
from pathlib import Path
import cv2
import numpy as np


# ---- 內嵌自 synth_cards.py ----
"""撲克牌合成器（整張牌分割與牌值分類兩個 Kaggle notebook 共用，以 `# @include synth_cards.py` 內嵌）。

把 gpiosenka 的單張牌照片以扇形／散放／橫排重疊貼到 DTD 紋理背景上：
- fan：手牌扇形，所有牌繞牌下方同一個樞紐點旋轉（最貼近手拿情境）
- scatter：散放、任意角度；row：橫排重疊（發牌）
之後加整體透視、手指遮擋（膚色橢圓）、模糊、亮度、雜訊、JPEG 壓縮。
render 回傳 (影像, 可見區域多邊形[正規化], 每張牌資訊)；可見面積 < 整張的 1% 的牌不出多邊形（實測常見只露出一條細邊＋牌角的後牌，要標成獨立的牌）。
"""
import glob
import math
import random
from pathlib import Path

import cv2
import numpy as np

S = 640                 # 合成圖邊長
# 合成器版本：3 = 現行（可見 1% 即標、4 成疊牌只露牌角細邊）；2 = seg-2 部署版的設定（可見 4%、無細邊疊牌）
SYNTH_V = globals().get("SYNTH_V", 3)


def find_dir(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits


def load_sources(card_root=None, bg_root=None):
    cards = [p for p in find_dir(f"{card_root or '/kaggle/input'}/**/train/*/*.jpg") if "joker" not in p.lower()]
    bgs = find_dir(f"{bg_root or '/kaggle/input'}/**/dtd/images/*/*.jpg") or \
          [p for p in find_dir(f"{bg_root or '/kaggle/input'}/**/*.jpg") if "texture" in p.lower() or "dtd" in p.lower()]
    return cards, bgs



CARD_AR = 1.4   # 牌高 / 牌寬


def rand_bg(bgs, rng):
    if bgs and rng.random() < 0.85:
        im = cv2.imread(rng.choice(bgs))
        if im is not None:
            h, w = im.shape[:2]
            s = rng.uniform(0.3, 1.0) * min(h, w)
            x, y = rng.uniform(0, w - s), rng.uniform(0, h - s)
            return cv2.resize(im[int(y):int(y + s), int(x):int(x + s)], (S, S))
    c = np.array([rng.randint(0, 255) for _ in range(3)], np.float32)
    noise = np.random.default_rng(rng.randint(0, 1 << 30)).normal(0, 12, (S, S, 3))
    return np.clip(c + noise, 0, 255).astype(np.uint8)


def card_quads(rng):
    n = rng.choice([1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 7])
    # stack：幾乎平行、往側邊錯開的疊牌（實測最常被分割合併成一塊的情況）
    mode = rng.choices(["fan", "scatter", "row", "stack"], [0.4, 0.15, 0.15, 0.3])[0]
    quads = []
    if mode == "stack":
        n = max(n, 2)
        h = rng.uniform(0.3, 0.65) * S
        w = h / CARD_AR
        ang = rng.uniform(-40, 40)
        # 每張往側邊露出的寬度：4 成是只露出牌角那一條的細邊（實測最常被併進前牌）
        shift = (rng.uniform(0.06, 0.15) if rng.random() < (0.4 if SYNTH_V >= 3 else 0) else rng.uniform(0.15, 0.5)) * w
        drop = rng.uniform(-0.08, 0.08) * h          # 每張上下微錯
        cx0, cy0 = rng.uniform(0.25, 0.6) * S, rng.uniform(0.3, 0.7) * S
        ca, sa = math.cos(math.radians(ang)), math.sin(math.radians(ang))
        for i in range(n):
            a = math.radians(ang + rng.uniform(-3, 3))
            local = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            dx, dy = i * shift, i * drop
            quads.append(local @ R.T + [cx0 + dx * ca - dy * sa, cy0 + dx * sa + dy * ca])
    elif mode == "fan":
        h = rng.uniform(0.28, 0.6) * S
        w = h / CARD_AR
        px, py = rng.uniform(0.25, 0.75) * S, rng.uniform(0.55, 0.95) * S
        base = rng.uniform(-35, 35)
        # 實際手牌常是很密的小角度扇形：一半機率 3~8°，一半 7~18°
        step = (rng.uniform(3, 8) if rng.random() < 0.5 else rng.uniform(7, 18)) * rng.choice([1, -1])
        off = rng.uniform(0.05, 0.25) * h
        a0 = base - step * (n - 1) / 2
        for i in range(n):
            a = math.radians(a0 + step * i)
            local = np.array([[-w / 2, -h + off], [w / 2, -h + off], [w / 2, off], [-w / 2, off]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            quads.append(local @ R.T + [px + rng.uniform(-3, 3), py + rng.uniform(-3, 3)])
    elif mode == "row":
        h = rng.uniform(0.2, 0.45) * S
        w = h / CARD_AR
        gap = rng.uniform(0.25, 0.7) * w
        cx0 = rng.uniform(0.15, 0.5) * S
        cy = rng.uniform(0.25, 0.75) * S
        ang = rng.uniform(-20, 20)
        for i in range(n):
            a = math.radians(ang + rng.uniform(-5, 5))
            local = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            quads.append(local @ R.T + [cx0 + i * gap * math.cos(math.radians(ang)), cy + i * gap * math.sin(math.radians(ang))])
    else:
        for _ in range(n):
            h = rng.uniform(0.15, 0.45) * S
            w = h / CARD_AR
            a = math.radians(rng.uniform(0, 360))
            local = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            quads.append(local @ R.T + [rng.uniform(0.15, 0.85) * S, rng.uniform(0.15, 0.85) * S])
    # 整體透視（模擬鏡頭角度）
    src = np.float32([[0, 0], [S, 0], [S, S], [0, S]])
    j = S * 0.08
    dst = np.float32([[rng.uniform(-j, j), rng.uniform(-j, j)], [S + rng.uniform(-j, j), rng.uniform(-j, j)],
                      [S + rng.uniform(-j, j), S + rng.uniform(-j, j)], [rng.uniform(-j, j), S + rng.uniform(-j, j)]])
    Hm = cv2.getPerspectiveTransform(src, dst)
    quads = [cv2.perspectiveTransform(q.reshape(-1, 1, 2).astype(np.float32), Hm).reshape(-1, 2) for q in quads]
    return quads, mode


def draw_distractors(img, rng):
    """背景負樣本：門、書櫃、畫框一類的長方形（不標註），教模型「長方形≠牌」。"""
    for _ in range(rng.choice([0, 0, 1, 1, 2, 3])):
        w, h = rng.uniform(0.1, 0.5) * S, rng.uniform(0.15, 0.8) * S
        x, y = rng.uniform(-0.1, 0.9) * S, rng.uniform(-0.1, 0.9) * S
        base = np.array(rng.choice([(200, 225, 235), (90, 140, 190), (230, 230, 225), (60, 90, 140), (180, 200, 210)]),
                        np.float32) + np.random.default_rng(rng.randint(0, 1 << 30)).uniform(-25, 25, 3)
        p1, p2 = (int(x), int(y)), (int(x + w), int(y + h))
        cv2.rectangle(img, p1, p2, tuple(int(v) for v in np.clip(base, 0, 255)), -1)
        kind = rng.random()
        if kind < 0.35:      # 書櫃：橫隔板＋書背
            for k in range(1, rng.randint(2, 5)):
                yy = int(y + h * k / 5)
                cv2.line(img, (p1[0], yy), (p2[0], yy), tuple(int(v * 0.6) for v in base), 3)
                xx = p1[0]
                while xx < p2[0]:
                    bw = rng.randint(4, 14)
                    cv2.rectangle(img, (xx, yy - rng.randint(10, 40)), (xx + bw, yy),
                                  tuple(rng.randint(30, 220) for _ in range(3)), -1)
                    xx += bw + rng.randint(0, 3)
        elif kind < 0.65:    # 畫框：內框＋雜訊內容
            m = int(min(w, h) * rng.uniform(0.08, 0.2))
            inner = img[max(0, p1[1] + m):max(0, p2[1] - m), max(0, p1[0] + m):max(0, p2[0] - m)]
            if inner.size:
                inner[:] = np.random.default_rng(rng.randint(0, 1 << 30)).integers(0, 255, inner.shape, dtype=np.uint8)
                inner[:] = cv2.GaussianBlur(inner, (0, 0), 6)
        else:                # 門：門框線＋門把
            cv2.rectangle(img, p1, p2, tuple(int(v * 0.7) for v in base), 2)
            cv2.circle(img, (int(x + w * 0.85), int(y + h * 0.5)), max(2, int(w * 0.03)), (60, 160, 200), -1)


def render(cards, bgs, rng):
    img = rand_bg(bgs, rng)
    if rng.random() < 0.6:
        draw_distractors(img, rng)
    quads, mode = card_quads(rng)
    masks = []
    full_areas = []
    infos = []   # 每張牌：gpiosenka 類別名（例 "ace of spades"）、整張外框四角、可見遮罩（與 masks 同一物件）
    for q in quads:
        path = rng.choice(cards)
        cim = cv2.imread(path)
        if cim is None:
            continue
        ch, cw = 280, 200
        cim = cv2.resize(cim, (cw, ch))
        M = cv2.getPerspectiveTransform(np.float32([[0, 0], [cw, 0], [cw, ch], [0, ch]]), q.astype(np.float32))
        warped = cv2.warpPerspective(cim, M, (S, S), flags=cv2.INTER_LINEAR)
        m = np.zeros((S, S), np.uint8)
        cv2.fillConvexPoly(m, q.astype(np.int32), 255)
        if masks and rng.random() < 0.6:   # 疊在別張牌上：牌緣外側一圈柔和陰影（真牌疊放時白牌之間唯一的邊界線索）
            sh = np.zeros((S, S), np.uint8)
            cv2.fillConvexPoly(sh, q.astype(np.int32), 255)
            sh = cv2.GaussianBlur(cv2.dilate(sh, np.ones((5, 5), np.uint8)), (0, 0), rng.uniform(1.5, 4))
            dark = 1 - rng.uniform(0.15, 0.4) * (sh.astype(np.float32) / 255)
            img[:] = (img * dark[..., None]).astype(np.uint8)
        img[m > 0] = warped[m > 0]
        if rng.random() < 0.4:   # 實際白牌疊白牌幾乎沒有輪廓線，只有一部分畫上細邊
            cv2.polylines(img, [q.astype(np.int32)], True, tuple(int(v) for v in rng.choice([(40, 40, 40), (200, 200, 200), (120, 120, 120)])), 1, cv2.LINE_AA)
        for pm in masks:
            pm[m > 0] = 0
        masks.append(m)
        full_areas.append(cv2.contourArea(q.astype(np.float32)))
        infos.append({"label": Path(path).parent.name, "quad": q, "mask": m})
    # 手指遮擋
    if mode == "fan" and rng.random() < 0.6 or rng.random() < 0.2:
        occ = np.zeros((S, S), np.uint8)
        for _ in range(rng.randint(1, 3)):
            cx, cy = rng.uniform(0.1, 0.9) * S, rng.uniform(0.5, 1.0) * S
            cv2.ellipse(occ, (int(cx), int(cy)), (int(rng.uniform(25, 70)), int(rng.uniform(50, 130))),
                        rng.uniform(0, 180), 0, 360, 255, -1)
        skin = np.array([rng.randint(60, 140), rng.randint(100, 180), rng.randint(150, 230)], np.uint8)
        img[occ > 0] = (img[occ > 0] * 0.15 + skin * 0.85).astype(np.uint8)
        for pm in masks:
            pm[occ > 0] = 0
    # 全域擾動
    alpha, beta = rng.uniform(0.6, 1.35), rng.uniform(-40, 40)
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    if rng.random() < 0.35:
        k = rng.choice([3, 5, 7])
        img = cv2.GaussianBlur(img, (k, k), 0) if rng.random() < 0.6 else cv2.blur(img, (k, 1))
    if rng.random() < 0.4:
        noise = np.random.default_rng(rng.randint(0, 1 << 30)).normal(0, rng.uniform(3, 12), img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    q = rng.randint(35, 95)
    img = cv2.imdecode(cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])[1], cv2.IMREAD_COLOR)
    # 標註：可見區域多邊形
    polys = []
    for m, fa in zip(masks, full_areas):
        vis = cv2.countNonZero(m)
        if fa <= 0 or vis < (0.01 if SYNTH_V >= 3 else 0.04) * fa:
            continue
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cs, key=cv2.contourArea)
        if cv2.contourArea(c) < max(0.008 * fa, 60):
            continue
        c = cv2.approxPolyDP(c, 0.004 * cv2.arcLength(c, True), True).reshape(-1, 2)
        if len(c) >= 3:
            polys.append(np.clip(c / S, 0, 1))
    return img, polys, infos
# ---- 內嵌結束 synth_cards.py ----
# ---- 內嵌自 ../blackjack_vision/upright.py ----
"""方案 B-2 的轉正裁角：整張牌旋轉框 → 透視轉正 → 裁左上與（轉 180° 的）右下牌角。

本檔只依賴 numpy／cv2、不用相對匯入：Kaggle 訓練 notebook 以 `# @include` 原樣內嵌，
確保訓練資料與本機推論的裁法完全一致。改這裡就要重新訓練 value_cls。
"""
import cv2
import numpy as np

# 牌角索引在轉正牌面中的位置（以牌高 H 為單位，實測：中心距最近角 (0.10H, 0.17H)，牌角框長邊約 0.22H）
INDEX_DX, INDEX_DY, INDEX_SIZE = 0.10, 0.17, 0.22
CROP_PAD = 0.25


def upright_card(frame, rbox):
    """把旋轉框透視轉正成直立牌面（短邊為寬、長邊為高，不鏡像）。回傳 (影像, 寬, 高) 或 None。
    方向只能確定到 180°（直立或倒立），所以牌角可能在左上或右下。"""
    p = np.array(rbox, np.float32)
    if len(p) != 4:
        return None
    # 確保來源點與目的點同為順時針（影像座標 y 向下），否則轉出來會是鏡像
    if (p[1, 0] - p[0, 0]) * (p[2, 1] - p[0, 1]) - (p[1, 1] - p[0, 1]) * (p[2, 0] - p[0, 0]) < 0:
        p = p[::-1].copy()
    if np.linalg.norm(p[1] - p[0]) > np.linalg.norm(p[2] - p[1]):   # 讓 p0→p1 為短邊（牌寬）
        p = np.roll(p, -1, axis=0)
    w, h = float(np.linalg.norm(p[1] - p[0])), float(np.linalg.norm(p[2] - p[1]))
    if w < 4 or h < 8:
        return None
    dst = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float32)
    img = cv2.warpPerspective(frame, cv2.getPerspectiveTransform(p, dst), (int(round(w)), int(round(h))),
                              borderMode=cv2.BORDER_REPLICATE)
    return img, w, h


def upright_corner_points(rbox):
    """兩個裁切中心在原圖上的座標（tl、br 順序同 upright_corner_crops），供合成資料判斷該處是否真有牌角。"""
    p = np.array(rbox, np.float32)
    if len(p) != 4:
        return []
    if (p[1, 0] - p[0, 0]) * (p[2, 1] - p[0, 1]) - (p[1, 1] - p[0, 1]) * (p[2, 0] - p[0, 0]) < 0:
        p = p[::-1].copy()
    if np.linalg.norm(p[1] - p[0]) > np.linalg.norm(p[2] - p[1]):
        p = np.roll(p, -1, axis=0)
    w, h = float(np.linalg.norm(p[1] - p[0])), float(np.linalg.norm(p[2] - p[1]))
    ex, ey = (p[1] - p[0]) / max(w, 1e-6), (p[3] - p[0]) / max(h, 1e-6)
    cx, cy = min(INDEX_DX * h, w / 2), INDEX_DY * h
    return [("tl", p[0] + ex * cx + ey * cy), ("br", p[0] + ex * (w - cx) + ey * (h - cy))]


def upright_to_frame(rbox, fx, fy):
    """轉正牌面上的相對座標 (fx＝x/w, fy＝y/h) 換回原圖座標（前端畫牌內切片命中位置用）。"""
    p = np.array(rbox, np.float32)
    if (p[1, 0] - p[0, 0]) * (p[2, 1] - p[0, 1]) - (p[1, 1] - p[0, 1]) * (p[2, 0] - p[0, 0]) < 0:
        p = p[::-1].copy()
    if np.linalg.norm(p[1] - p[0]) > np.linalg.norm(p[2] - p[1]):
        p = np.roll(p, -1, axis=0)
    q = p[0] + (p[1] - p[0]) * fx + (p[3] - p[0]) * fy
    return [round(float(q[0]), 1), round(float(q[1]), 1)]


def face_white_ratio(img):
    """轉正牌面中「低彩度、高亮度」像素的比例（牌面白底）。背景框（書櫃、海報）通常 ≤ 0.55。"""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return float(((hsv[..., 1] < 60) & (hsv[..., 2] > 140)).mean())


def face_border_white(img, inner=0.03, outer=0.10):
    """轉正牌面「邊框帶」（距邊緣 inner～outer 倍牌寬）的白底比例。撲克牌（含人頭牌、牌背）四周都有白邊，
    海報、畫框、書背的邊緣通常是深色或彩色。略過最外圈 inner，避開分割框邊緣混到的背景。"""
    h, w = img.shape[:2]
    a, b = max(1, int(inner * w)), max(2, int(outer * w))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    white = (hsv[..., 1] < 60) & (hsv[..., 2] > 140)
    band = np.zeros((h, w), bool)
    band[a:h - a, a:w - a] = True
    band[b:h - b, b:w - b] = False
    return float(white[band].mean()) if band.any() else 0.0


def face_color_ratio(img):
    """轉正牌面中「飽和、非紅色」像素的比例。撲克牌只有紅黑兩色加白底（人頭牌的黃藍綠面積很小，實拍 ≤ 0.07），
    書背、海報、抱枕通常大片彩色（實拍書櫃 0.29～0.43）。"""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hue = hsv[..., 0]
    colored = (hsv[..., 1] > 70) & (hsv[..., 2] > 50) & (hue > 12) & (hue < 160)
    return float(colored.mean())


def scan_positions(w, h, step=0.5):
    """牌內切片的裁切中心（轉正牌面座標）與方向：沿上緣橫向、沿左緣縱向各掃一排（正向），
    再沿下緣、右緣掃一排（轉 180°）。分割把扇形／錯開疊牌框成一塊時，後面那張牌的牌角索引就落在這些邊上。
    step 以牌角框邊長為單位。tl、br 兩個標準位置不在內（由 upright_corner_crops 負責）。"""
    cx, cy, s = min(INDEX_DX * h, w / 2), INDEX_DY * h, INDEX_SIZE * h * step
    xs = [cx + s * k for k in range(1, int((w - 2 * cx) / s) + 1)]
    ys = [cy + s * k for k in range(1, int((h - 2 * cy) / s) + 1)]
    out = [((x, cy), False) for x in xs] + [((cx, y), False) for y in ys]
    out += [((w - x, h - cy), True) for x in xs] + [((w - cx, h - y), True) for y in ys]
    return out


def upright_scan_crops(frame, rbox, pad=CROP_PAD, step=0.5):
    """牌內切片：轉正後沿四邊掃描裁切（大小同 upright_corner_crops），回傳 [((x, y), 是否轉180°, 影像)]。"""
    u = upright_card(frame, rbox)
    if u is None:
        return []
    img, w, h = u
    side = INDEX_SIZE * h * (1 + 2 * pad)
    big = cv2.copyMakeBorder(img, int(side), int(side), int(side), int(side), cv2.BORDER_REPLICATE)
    out = []
    for (x, y), flip in scan_positions(w, h, step):
        a, b = int(x - side / 2 + side), int(y - side / 2 + side)
        c = big[b:b + int(side), a:a + int(side)]
        if c.size and min(c.shape[:2]) >= 6:
            out.append(((x, y), flip, cv2.rotate(c, cv2.ROTATE_180) if flip else c))
    return out


def upright_corner_crops(frame, rbox, pad=CROP_PAD):
    """轉正後裁兩個牌角（正方形，邊長＝牌角框 ×(1+2·pad)）：
    tl＝左上原樣、br＝右下轉 180°（倒立的牌或扇形窄條，牌角會落在這裡）。"""
    u = upright_card(frame, rbox)
    if u is None:
        return []
    img, w, h = u
    side = INDEX_SIZE * h * (1 + 2 * pad)
    cx = min(INDEX_DX * h, w / 2)    # 扇形窄條比牌角還窄時，取窄條中線
    cy = INDEX_DY * h
    big = cv2.copyMakeBorder(img, int(side), int(side), int(side), int(side), cv2.BORDER_REPLICATE)
    out = []
    for name, (x, y) in (("tl", (cx, cy)), ("br", (w - cx, h - cy))):
        a, b = int(x - side / 2 + side), int(y - side / 2 + side)
        c = big[b:b + int(side), a:a + int(side)]
        if c.size and min(c.shape[:2]) >= 6:
            out.append((name, c if name == "tl" else cv2.rotate(c, cv2.ROTATE_180)))
    return out
# ---- 內嵌結束 ../blackjack_vision/upright.py ----
# ---- 內嵌自 value_cls_data.py ----
"""value_cls 訓練資料：合成場景 → 轉正裁角 → 標牌值或 none。train_value_cls.py 與 train_value_ft.py 共用（# @include）。
需要先定義 IMGSZ、NONE_RATIO、SEED，並內嵌 synth_cards.py、upright.py。"""

## 標註裁角
牌照片貼上去的四角 `quad` 依序是牌的左上、右上、右下、左下，所以真正牌角（左上、右下）的位置可由同一個透視算出。
裁切中心離真牌角 ≤ 0.12×牌高、且該處沒被蓋住 → 標牌值；否則 → `none`。

In [ ]:
RANKS = {"ace": "A", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8",
         "nine": "9", "ten": "10", "jack": "J", "queen": "Q", "king": "K"}
SUITS = {"clubs": "c", "diamonds": "d", "hearts": "h", "spades": "s"}


def to_label(name: str):
    parts = name.lower().split(" of ")
    if len(parts) != 2 or parts[0] not in RANKS or parts[1] not in SUITS:
        return None
    return RANKS[parts[0]] + SUITS[parts[1]]


def true_index_points(q):
    """牌照片座標 → 原圖：左上、右下兩個牌角索引中心。"""
    cw, ch = 200.0, 280.0   # 與 render 貼圖尺寸一致
    M = cv2.getPerspectiveTransform(np.float32([[0, 0], [cw, 0], [cw, ch], [0, ch]]), q.astype(np.float32))
    pts = np.float32([[[INDEX_DX * ch, INDEX_DY * ch]], [[cw - INDEX_DX * ch, ch - INDEX_DY * ch]]])
    return cv2.perspectiveTransform(pts, M).reshape(-1, 2), float(np.linalg.norm(q[3] - q[0]))


def visible_at(mask, pt, r):
    x, y, r = int(pt[0]), int(pt[1]), max(2, int(r))
    win = mask[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1]
    return win.size > 0 and (win > 0).mean() >= 0.6


def crops_from_scene(img, infos):
    out = []
    for inf in infos:
        lab = to_label(inf["label"])
        m = inf["mask"]
        if lab is None or cv2.countNonZero(m) < 400:
            continue
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cs, key=cv2.contourArea).reshape(-1, 2).astype(np.float32)
        rbox = cv2.boxPoints(cv2.minAreaRect(c))                     # 與本機 CardSegmenter 相同的旋轉框
        idx_pts, card_h = true_index_points(inf["quad"])
        centers = dict(upright_corner_points(rbox))
        for name, crop in upright_corner_crops(img, rbox):
            cpt = centers[name]
            hit = any(np.linalg.norm(cpt - t) <= 0.12 * card_h and visible_at(m, t, 0.04 * card_h) for t in idx_pts)
            out.append((lab if hit else "none", cv2.resize(crop, (IMGSZ, IMGSZ))))
    return out


def build(cards, bgs, out_dir: Path, n_train, n_val, seed=SEED):
    stats = {}
    for split, n in (("train", n_train), ("val", n_val)):
        rng = random.Random(seed + (0 if split == "train" else 999))
        buf, cnt = [], Counter()
        for _ in range(n):
            img, _, infos = render(cards, bgs, rng)
            for lab, crop in crops_from_scene(img, infos):
                buf.append((lab, crop)); cnt[lab] += 1
        avg = np.mean([v for k, v in cnt.items() if k != "none"])
        keep_none = int(avg * NONE_RATIO)
        rng.shuffle(buf)
        written = Counter()
        for lab, crop in buf:
            if lab == "none" and written["none"] >= keep_none:
                continue
            d = out_dir / split / lab
            d.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(d / f"{written[lab]:05d}.jpg"), crop)
            written[lab] += 1
        stats[split] = dict(written)
        print(split, "類別數", len(written), "總數", sum(written.values()), "none", written["none"],
              "最少", min(v for k, v in written.items() if k != "none"))
    return stats
# ---- 內嵌結束 value_cls_data.py ----

In [ ]:
def find_real():
    for p in Path("/kaggle/input").rglob("real"):
        if p.is_dir() and any(p.iterdir()):
            return p, next(Path("/kaggle/input").rglob(f"{BASE_DIR}/best.pt"))
    raise FileNotFoundError("找不到 cards-real-crops（real/、base/ 或 base26/best.pt，依 BASE_DIR）")


def add_real(real: Path, ds: Path):
    """真牌裁圖縮到 IMGSZ 後複製進 train/<類別>/：牌值類別過採樣到約 REAL_POS_TARGET 張、none ×REAL_NONE_REPEAT。"""
    cnt = Counter()
    for d in sorted(real.iterdir()):
        files = sorted(d.glob("*.jpg"))
        rep = REAL_NONE_REPEAT if d.name == "none" else min(20, max(1, round(REAL_POS_TARGET / len(files))))
        out = ds / "train" / d.name
        out.mkdir(parents=True, exist_ok=True)
        for f in files:
            img = cv2.resize(cv2.imread(str(f)), (IMGSZ, IMGSZ))
            for k in range(rep):
                cv2.imwrite(str(out / f"real_{f.stem}_{k}.jpg"), img)
                cnt[d.name] += 1
    print("真牌加入", dict(cnt))
    return dict(cnt)


if os.path.exists("/kaggle/input"):
    cards, bgs = load_sources()
    print("牌照片", len(cards), "背景", len(bgs))
    REAL, BASE = find_real()
    DS = Path("/tmp/value_ft")
    if not (DS / "train").exists():
        STATS = build(cards, bgs, DS, N_TRAIN, N_VAL)
        REAL_STATS = add_real(REAL, DS)

## 微調
從現有權重出發、學習率調低。訓練前後都在同一個合成驗證集上量 top-1，確認合成能力沒有明顯退步。

In [ ]:
if os.path.exists("/kaggle/input"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics>=8.4.0"], check=True)
    from ultralytics import YOLO
    before = YOLO(str(BASE)).val(data=str(DS), imgsz=IMGSZ, batch=BATCH, device=0)
    model = YOLO(str(BASE))
    model.train(data=str(DS), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, workers=4,
                optimizer="AdamW", lr0=LR0, warmup_epochs=1, fliplr=0.0, degrees=8, patience=6, seed=SEED,
                project="/kaggle/working/runs", name="value_ft", exist_ok=True, plots=True)
    run = Path("/kaggle/working/runs/value_ft")
    best = YOLO(str(run / "weights" / "best.pt"))
    r = best.val(data=str(DS), imgsz=IMGSZ, batch=BATCH, device=0)
    metrics = {"base": BASE_DIR, "top1": float(r.top1), "top5": float(r.top5),
               "synth_val_top1_before": float(before.top1), "imgsz": IMGSZ, "epochs": EPOCHS, "lr0": LR0,
               "real_pos_target": REAL_POS_TARGET, "real_none_repeat": REAL_NONE_REPEAT, "real": REAL_STATS,
               "nb_tag": NB_TAG}
    print(metrics)
    OUT = Path("/kaggle/working/out") / OUT_NAME; OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy(run / "weights" / "best.pt", OUT / "best.pt")
    for f in ("results.csv", "results.png", "confusion_matrix_normalized.png"):
        if (run / f).exists():
            shutil.copy(run / f, OUT / f)
    json.dump(metrics, open(OUT / "metrics.json", "w"), indent=2)
    shutil.make_archive("/kaggle/working/cards_value_ft", "zip", "/kaggle/working/out")
    print("下載 cards_value_ft.zip，解壓到本機專案的 models/ 底下")